# Assignment 1A — Part B
Create grounded instruction pairs, train QLoRA adapters A/B/C, and compare the same three banking prompts.

### 1. Clone the repository

In [ ]:
from pathlib import Path
import subprocess
import os

REPO_URL = "https://github.com/<YOUR_USERNAME>/<YOUR_REPOSITORY>.git"
PROJECT = Path("/content/banking-compliance-llm-assignment-1a")

if not (PROJECT / "src").exists():
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT)], check=True)

os.chdir(PROJECT)

### Install dependencies:

In [ ]:
!pip install -q -r requirements.txt

### 2. Mount Drive and restore Part A files

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path
import shutil

SAVE_DIR = Path("/content/drive/MyDrive/LLM_ASSIGNMENT_PART_A_RESULTS")

items = [
    "data/cleaned_txt",
    "data/train_corpus",
    "data/eval_corpus",
    "data/train_packed.parquet",
    "reports",
    "results",
    "models/cpt_model",
]

for item in items:
    source = SAVE_DIR / item
    destination = PROJECT / item

    if source.is_dir():
        shutil.copytree(source, destination, dirs_exist_ok=True)
    elif source.is_file():
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, destination)

print("Part A files restored.")

### Verify:

In [ ]:
!ls models/cpt_model
!find data/cleaned_txt -type f -iname "*.txt" | wc -l

### 3. Create the instruction dataset

In [ ]:
!python -m src.data.create_instruction_dataset --minimum 100

### Verify:

In [ ]:
import json

print(json.dumps(
    json.loads(
        Path("data/instruction_dataset/split_report.json").read_text()
    ),
    indent=2
))

### 4. Train adapters A, B, and C

In [ ]:
!python -m src.qlora.train_adapter_A --max-steps 500

In [ ]:
!python -m src.qlora.train_adapter_B --max-steps 500

In [ ]:
!python -m src.qlora.train_adapter_C --max-steps 500

### 5. Compare adapters

In [ ]:
!python -m src.evaluation.adapter_comparison
!python -m src.evaluation.build_report

In [ ]:
pd.read_csv("results/adapter_comparison/adapter_comparison.csv")

### Save Part B outputs to Google Drive

In [ ]:
SAVE_DIR = Path("/content/drive/MyDrive/LLM_ASSIGNMENT_PART_B_RESULTS")

items = [
    "data/instruction_dataset",
    "reports",
    "results",
    "models/adapter_A",
    "models/adapter_B",
    "models/adapter_C",
]

for item in items:
    source = PROJECT / item
    destination = SAVE_DIR / item

    if source.is_dir():
        shutil.copytree(source, destination, dirs_exist_ok=True)
    elif source.is_file():
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, destination)

print("Part B saved to:", SAVE_DIR)